# 🚀 De la Computadora al Mundo: Introducción al Despliegue (Deployment)

Hasta ahora, hemos construido nuestro Sistema de Fiscalización Inteligente en un entorno "local" (nuestras propias computadoras). Funciona perfecto, pero tiene un problema fundamental: **nadie más puede usarlo**. Si el Ministro quiere ver el tablero, tendría que sentarse físicamente en nuestra computadora.

Para que una herramienta tenga impacto real en la gestión pública, debe estar accesible en la web. A este proceso de llevar nuestra aplicación desde nuestra máquina a un servidor público se lo conoce como **Despliegue o Deployment**.

## 🏗️ ¿Qué significa hacer un "Deploy"?

Imaginemos que escribimos un libro. Mientras está en nuestro documento de Word, solo nosotros podemos leerlo (entorno local). Hacer el *deploy* equivale a llevar ese libro a una imprenta y distribuirlo en bibliotecas (servidores) para que cualquier ciudadano pueda consultarlo a través de una dirección específica (URL).

Cuando hacemos un deploy, lo que realmente estamos haciendo es:
1. Alquilar una computadora (servidor) que está encendida las 24 horas.
2. Pasarle nuestro código a esa computadora.
3. Instalarle las herramientas necesarias (Python, librerías) para que entienda nuestro código.
4. Conectar esa computadora a internet con una dirección pública.

## 🏛️ La Arquitectura de nuestro Despliegue

En un entorno corporativo tradicional, alquilaríamos un servidor gigante y pondríamos todo ahí adentro. Pero hoy en día, la mejor práctica (y la más económica) es usar servicios especializados para cada parte de nuestro sistema. 

Para este proyecto, vamos a usar una arquitectura de **Cero Configuración de Infraestructura**:

* **El Cerebro (Backend/FastAPI):** Lo alojaremos en **Render**. Es una plataforma excelente para mantener nuestra API funcionando y escuchando peticiones.
* **La Cara Visible (Frontend/Streamlit):** Lo alojaremos en **Streamlit Community Cloud**, un servicio oficial diseñado específicamente para que los tableros de datos funcionen de forma rápida y gratuita.

## ⚠️ El factor efímero: ¿Qué pasa con nuestra Base de Datos?

Nuestro sistema actual usa **SQLite** (`ministerio.db`). SQLite es fantástico porque guarda toda la información en un simple archivo de texto en nuestra carpeta. 

Sin embargo, al usar servidores gratuitos en la nube (como Render), nos enfrentamos a una regla estricta: **Los discos duros son efímeros.**

¿Qué significa esto? 
Para ahorrar costos, si nuestra API no recibe visitas durante 15 minutos, Render la "pone a dormir". Cuando alguien vuelve a entrar, Render no despierta la máquina original, sino que crea un "clon" totalmente nuevo desde nuestro repositorio de GitHub. 

**El resultado:** Todo lo que hayamos guardado en el archivo `ministerio.db` durante esa sesión se borrará y volveremos a empezar con una base de datos vacía. 

### ¿Es esto un problema?
Depende del objetivo:
* **Escenario A (Laboratorio / Pruebas):** Si queremos probar el flujo del algoritmo, cargar datos de prueba, ver los gráficos y cerrar, **es una ventaja**. Cada vez que entramos tenemos un sistema limpio. (Este es el camino que usaremos hoy).
* **Escenario B (Sistema Real):** Si necesitamos que los datos del Estado no se borren (persistencia real), no podemos usar SQLite. Deberíamos conectar nuestro código a una base de datos externa en la nube (como PostgreSQL usando servicios como Neon.tech o Supabase).

# 🛠️ Manos a la obra: Guía de Despliegue

Vamos a subir nuestro proyecto a la web paso a paso. 

## Paso 0: Preparativos en nuestro código

Antes de subir nada a la nube, la computadora del servidor necesita saber dos cosas: qué librerías tiene que instalar y a dónde tiene que conectarse.

**1. El archivo de dependencias (`requirements.txt`)**
Debemos crear un archivo de texto en la carpeta de nuestro proyecto llamado EXACTAMENTE `requirements.txt`. Adentro, listaremos las herramientas que usamos:
```text
fastapi
uvicorn
streamlit
pandas
requests
pydantic
sqlalchemy
python-dotenv

## 🌉 Paso 1: El Puente (Subir el código a GitHub)

Los servidores en la nube no pueden entrar a nuestras computadoras a buscar el código. Necesitan un lugar público y seguro de donde descargarlo. Ese lugar es **GitHub**.

1. Creen una cuenta gratuita en [GitHub](https://github.com).
2. Hagan clic en el botón verde **New** para crear un nuevo "Repositorio" (que es básicamente una carpeta en la nube).
3. Pónganle un nombre (ej. `sistema-fiscalizacion`), déjenlo como **Public** y hagan clic en "Create repository".
4. Suban todos los archivos del proyecto (`main.py`, `frontend.py`, `database.py`, `estrategias.py`, `esquemas.py` y muy importante, el `requirements.txt`).
5. **REGLA DE ORO:** NUNCA suban el archivo `.env`. Ese archivo tiene nuestras claves de seguridad y si lo suben, cualquier persona en internet podría verlas. Las variables de seguridad las configuraremos directamente en el servidor más adelante.

## ⚙️ Paso 2: Encender el "Cerebro" (Backend en Render)

Ahora vamos a alquilar nuestra primera computadora gratuita para que corra el archivo `main.py`.

1. Entren a [Render.com](https://render.com) e inicien sesión usando su cuenta de GitHub.
2. En el panel principal, hagan clic en **New** y elijan **Web Service**.
3. Seleccionen la opción "Build and deploy from a Git repository" y conecten el repositorio que acaban de crear.
4. Completen la configuración técnica:
   * **Language:** Python 3
   * **Start Command:** `uvicorn main:app --host 0.0.0.0 --port 10000`
     *(Explicación técnica: Poner `0.0.0.0` es vital. Le dice a nuestra API que deje de ser local y acepte conexiones provenientes de cualquier parte de internet).*
   * **Instance Type:** Free
5. **Configurar la Seguridad (Environment Variables):** Como no subimos el archivo `.env`, debemos darle las contraseñas al servidor manualmente. Hagan clic en "Advanced" o "Environment Variables" y agreguen:
   * Key: `TOKEN_MINISTERIAL` | Value: `ClaveSecreta123`
   * Key: `TOKEN_MINISTERIAL_ZONA_1` | Value: `123`
6. Hagan clic en **Create Web Service**. 

Render tardará unos minutos en instalar todo. Cuando termine, verán un enlace verde en la parte superior izquierda (algo como `https://tu-proyecto.onrender.com`). **Copien ese enlace, lo vamos a necesitar.**

## 📍 Paso 3: El "Cambio de Domicilio" (Actualizar el Frontend)

Este es el error número uno al hacer despliegues. En este momento, nuestro archivo `frontend.py` tiene una línea que dice:
`url_api = "http://127.0.0.1:8080/priorizar/..."`

En informática, `127.0.0.1` significa "buscá en esta misma computadora". Si subimos el tablero a la nube con esa dirección, el tablero va a intentar buscar al servidor dentro de sí mismo y va a fallar ("Conexión Rechazada"). 

Tenemos que avisarle al Frontend que nuestro Backend se mudó de casa.

1. Abran su archivo `frontend.py` (pueden hacerlo directamente editándolo en GitHub con el ícono del lápiz).
2. Busquen la línea de la URL y reemplacen `http://127.0.0.1:8080` por el enlace público que les acaba de dar Render.
   * *Quedará algo así:* `url_api = f"https://tu-proyecto.onrender.com/priorizar/{estrategia_elegida}"`
3. Guarden los cambios (Hagan un *Commit* en GitHub).

## 📊 Paso 4: La "Cara Visible" (Frontend en Streamlit)

Con el motor funcionando en Render y el tablero apuntando a la dirección correcta, solo nos falta poner la interfaz gráfica en internet.

1. Entren a [share.streamlit.io](https://share.streamlit.io) e inicien sesión con GitHub.
2. Hagan clic en el botón **New app**.
3. Completen el formulario para decirle a Streamlit dónde está el código:
   * **Repository:** Seleccionen el repositorio de su proyecto.
   * **Branch:** `main` (o `master`).
   * **Main file path:** Escriban `frontend.py`.
4. Hagan clic en **Deploy**.

Streamlit leerá el archivo `requirements.txt`, instalará las librerías gráficas y en un par de minutos la pantalla de carga desaparecerá. 
